# ds004752 V5.6 Tranche 2.4 Feature Matrix Materializer Skeleton

Purpose: record a claim-closed materializer skeleton before real feature values are written.

Integrity boundary: this notebook does not read EDF payloads, materialize feature values, train models, run comparators, compute efficacy metrics, or open claims.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or update repo

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'

def run(cmd, cwd=None):
    print('$', ' '.join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', 'main'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO_DIR)

## 3. Confirm Tranche 2.4 code is present

In [ ]:
%cd /content/eeg-ds004752
!git log --oneline -8

from pathlib import Path

required_files = [
    Path('src/v56/feature_matrix_materializer_skeleton.py'),
    Path('configs/v56/feature_matrix_materializer_skeleton.json'),
    Path('tests/unit/test_v56_feature_matrix_materializer_skeleton.py'),
    Path('bootstrap/run_v56_feature_matrix_materializer_skeleton.sh'),
    Path('docs/26_v56_feature_matrix_materializer_skeleton_runbook_2026-05-04.md'),
]
missing = [str(p) for p in required_files if not p.exists()]
print({'missing_required_files': missing})
assert not missing, missing

cli_text = Path('src/cli.py').read_text(encoding='utf-8')
assert 'v56-feature-matrix-materializer-skeleton' in cli_text
assert 'run_v56_feature_matrix_materializer_skeleton' in cli_text
print('V5.6 Tranche 2.4 materializer skeleton code is present.')

## 4. Configure source-of-truth paths

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/20260424T100159866284Z'
SPLIT_LOCK_RUN = DRIVE_ROOT / 'artifacts/v56_split_registry_lock/latest.txt'
FEATURE_PROVENANCE_RUN = DRIVE_ROOT / 'artifacts/v56_feature_provenance_populated/latest.txt'
FEATURE_MATRIX_PLAN_RUN = DRIVE_ROOT / 'artifacts/v56_feature_matrix_plan/latest.txt'
LEAKAGE_AUDIT_PLAN_RUN = DRIVE_ROOT / 'artifacts/v56_feature_matrix_leakage_audit_plan/latest.txt'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/v56_feature_matrix_materializer_skeleton'

for path in [GATE0_RUN, SPLIT_LOCK_RUN, FEATURE_PROVENANCE_RUN, FEATURE_MATRIX_PLAN_RUN, LEAKAGE_AUDIT_PLAN_RUN]:
    print(path, path.exists())
    assert path.exists(), f'Missing required path: {path}'
print('OUTPUT_ROOT =', OUTPUT_ROOT)

## 5. Preflight upstream artifacts

In [ ]:
import json

feature_plan_dir = Path(FEATURE_MATRIX_PLAN_RUN.read_text().strip())
leakage_dir = Path(LEAKAGE_AUDIT_PLAN_RUN.read_text().strip())
feature_plan = json.loads((feature_plan_dir / 'v56_feature_matrix_plan.json').read_text())
leakage_plan = json.loads((leakage_dir / 'v56_feature_matrix_leakage_audit_plan.json').read_text())
leakage_validation = json.loads((leakage_dir / 'v56_feature_matrix_leakage_audit_plan_validation.json').read_text())

preflight = {
    'feature_plan_status': feature_plan.get('status'),
    'leakage_status': leakage_plan.get('status'),
    'leakage_validation_status': leakage_validation.get('status'),
    'claim_closed': leakage_plan.get('claim_closed'),
    'feature_matrix_materialized': leakage_plan.get('scientific_boundary', {}).get('feature_matrix_materialized'),
    'runtime_comparator_logs_audited': leakage_plan.get('scientific_boundary', {}).get('runtime_comparator_logs_audited'),
}
print(json.dumps(preflight, indent=2))
assert preflight['feature_plan_status'] == 'planned_feature_matrix_contract_recorded', preflight
assert preflight['leakage_status'] == 'planned_feature_matrix_leakage_audit_recorded', preflight
assert preflight['leakage_validation_status'] == 'v56_feature_matrix_leakage_audit_plan_validation_passed', preflight
assert preflight['claim_closed'] is True, preflight
assert preflight['feature_matrix_materialized'] is False, preflight
assert preflight['runtime_comparator_logs_audited'] is False, preflight
print('Preflight passed: Tranche 2.2/2.3 artifacts are ready.')

## 6. Run V5.6 Tranche 2.4 materializer skeleton

In [ ]:
%cd /content/eeg-ds004752
import subprocess

cmd = [
    'bash', 'bootstrap/run_v56_feature_matrix_materializer_skeleton.sh',
    str(GATE0_RUN), str(SPLIT_LOCK_RUN), str(FEATURE_PROVENANCE_RUN),
    str(FEATURE_MATRIX_PLAN_RUN), str(LEAKAGE_AUDIT_PLAN_RUN), str(OUTPUT_ROOT),
]
print('$', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print('returncode:', result.returncode)
print('--- stdout ---')
print(result.stdout)
print('--- stderr ---')
print(result.stderr)
assert result.returncode == 0, result.stderr or result.stdout
latest_after_run = OUTPUT_ROOT / 'latest.txt'
print('latest_after_run_exists:', latest_after_run.exists(), latest_after_run)
assert latest_after_run.exists(), latest_after_run

## 7. Load latest materializer skeleton artifact

In [ ]:
latest = OUTPUT_ROOT / 'latest.txt'
if not latest.exists():
    print('Missing latest pointer:', latest)
    print('OUTPUT_ROOT exists:', OUTPUT_ROOT.exists())
    if OUTPUT_ROOT.exists():
        print('OUTPUT_ROOT children:', sorted(str(p) for p in OUTPUT_ROOT.iterdir())[:20])
    raise AssertionError(latest)
run_dir = Path(latest.read_text().strip())
assert run_dir.exists(), run_dir

skeleton = json.loads((run_dir / 'v56_feature_matrix_materializer_skeleton.json').read_text())
summary = json.loads((run_dir / 'v56_feature_matrix_materializer_skeleton_summary.json').read_text())
validation = json.loads((run_dir / 'v56_feature_matrix_materializer_skeleton_validation.json').read_text())

compact = {
    'run_dir': str(run_dir),
    'status': skeleton.get('status'),
    'validation_status': validation.get('status'),
    'claim_closed': skeleton.get('claim_closed'),
    'claim_ready': skeleton.get('claim_ready'),
    'edf_payloads_read': skeleton.get('scientific_boundary', {}).get('edf_payloads_read'),
    'feature_matrix_materialized': skeleton.get('scientific_boundary', {}).get('feature_matrix_materialized'),
    'feature_values_written': skeleton.get('scientific_boundary', {}).get('feature_values_written'),
    'model_training_run': skeleton.get('scientific_boundary', {}).get('model_training_run'),
    'efficacy_metrics_computed': skeleton.get('scientific_boundary', {}).get('efficacy_metrics_computed'),
}
print(json.dumps(compact, indent=2))

## 8. Integrity assertions

In [ ]:
assert skeleton['status'] == 'planned_feature_matrix_materializer_skeleton_recorded', skeleton
assert validation['status'] == 'v56_feature_matrix_materializer_skeleton_validation_passed', validation
assert validation['blocking_errors'] == [], validation
assert skeleton['claim_closed'] is True, skeleton
assert skeleton['claim_ready'] is False, skeleton
assert skeleton['scientific_boundary']['edf_payloads_read'] is False, skeleton
assert skeleton['scientific_boundary']['feature_matrix_materialized'] is False, skeleton
assert skeleton['scientific_boundary']['feature_values_written'] is False, skeleton
assert skeleton['scientific_boundary']['model_training_run'] is False, skeleton
assert skeleton['scientific_boundary']['comparator_execution_run'] is False, skeleton
assert skeleton['scientific_boundary']['efficacy_metrics_computed'] is False, skeleton
assert summary['claim_closed'] is True, summary
assert summary['edf_payloads_read'] is False, summary
assert summary['feature_matrix_materialized'] is False, summary
assert summary['feature_values_written'] is False, summary
print('V5.6 Tranche 2.4 materializer skeleton passed integrity assertions.')

## 9. Decision gate closeout

In [ ]:
closeout = {
    'tranche2_4_status': 'feature_matrix_materializer_skeleton_recorded',
    'run_dir': str(run_dir),
    'claim_closed': True,
    'claim_ready': False,
    'edf_payloads_read': False,
    'feature_matrix_materialized': False,
    'feature_values_written': False,
    'model_training_run': False,
    'comparator_execution_run': False,
    'efficacy_metrics_computed': False,
    'next_step': 'manual_review_then_implement_real_scalp_feature_matrix_materializer',
    'not_allowed_next': ['do_not_train_RIFT_Net_Lite_yet', 'do_not_run_A3_A4_comparators_yet', 'do_not_open_efficacy_claim'],
}
print(json.dumps(closeout, indent=2))
print('\n=== report preview ===')
print((run_dir / 'v56_feature_matrix_materializer_skeleton_report.md').read_text()[:2500])